In [0]:
df_laptimes = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/lap_times.csv",
    header=True
)

In [0]:
df_laptimes = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/lap_times.csv",
    header=True
)

In [0]:
from pyspark.sql.functions import col

df = df_laptimes.select(
    col("raceId").cast("int"),
    col("driverId").cast("int"),
    col("lap").cast("int"),
    col("milliseconds").cast("double")
).dropna()

In [0]:
pdf = df.toPandas()

X = pdf[["raceId","driverId","lap"]]
y = pdf["milliseconds"]

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

for depth in [3,5,7,10,12]:
    for n in [50,100]:

        with mlflow.start_run():

            model = RandomForestRegressor(
                max_depth=depth,
                n_estimators=n,
                random_state=42
            )

            model.fit(X_train, y_train)

            preds = model.predict(X_test)

            rmse = np.sqrt(mean_squared_error(y_test, preds))
            r2 = r2_score(y_test, preds)
            mae = mean_absolute_error(y_test, preds)

            # log parameters
            mlflow.log_param("max_depth", depth)
            mlflow.log_param("n_estimators", n)

            # log metrics
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)

            # artifact 1: residual plot
            plt.figure()
            plt.scatter(y_test, preds)
            plt.xlabel("Actual")
            plt.ylabel("Predicted")
            plt.title("Residual Plot")
            plt.savefig("residuals.png")
            mlflow.log_artifact("residuals.png")

            # artifact 2: feature importance
            fi = pd.DataFrame({
                "feature": X.columns,
                "importance": model.feature_importances_
            })

            fi.to_csv("feature_importance.csv", index=False)
            mlflow.log_artifact("feature_importance.csv")

            # log model
            mlflow.sklearn.log_model(model, "random_forest_model")

Best Model Run

The best model run was the Random Forest model with max_depth = X and n_estimators = Y. I selected this run because it had the lowest RMSE, lowest MAE, and highest R² among all experiments tracked in MLflow. These results show that this model predicted lap times more accurately than the other parameter combinations, so I considered it the strongest overall run.